# 13 — Agentic Forecasting

**polars-ts** provides a multi-agent forecasting framework inspired by
[TimeSeriesScientist (2025)](https://arxiv.org/abs/2510.01538) and
[DCATS (Yeh et al., 2025)](https://arxiv.org/abs/2508.04231).

The pipeline chains four specialized agents:

| Agent | Role |
|-------|------|
| **CuratorAgent** | Data diagnostics, missing-value imputation, outlier clipping, stationarity checks, lookback-window recommendation |
| **PlannerAgent** | Heuristic (or LLM-guided) model-candidate selection based on data characteristics |
| **ForecasterAgent** | Train/validation split, per-candidate MAE scoring, best-model or weighted-ensemble selection |
| **ReporterAgent** | Structured Markdown report synthesizing all stages |

`TimeSeriesScientist` orchestrates the full chain in a single `.run()` call.

In [ ]:
import datetime

import numpy as np
import polars as pl

# Build a synthetic daily series with trend + weekly seasonality + noise
n = 180
dates = [datetime.date(2023, 1, 1) + datetime.timedelta(days=i) for i in range(n)]
rng = np.random.default_rng(42)
trend = np.linspace(10, 60, n)
seasonal = 8 * np.sin(2 * np.pi * np.arange(n) / 7)
noise = rng.normal(0, 2, n)
values = trend + seasonal + noise

# Inject a few data-quality issues
values[25] = float("nan")
values[80] = 200.0  # outlier

df = pl.DataFrame({"unique_id": ["demo"] * n, "ds": dates, "y": values.tolist()})
df.head()

## 13.1 One-shot pipeline with `TimeSeriesScientist`

The simplest usage: pass a DataFrame and get predictions + a Markdown report.

In [ ]:
from polars_ts.agents import TimeSeriesScientist

scientist = TimeSeriesScientist(horizon=14)
result = scientist.run(df)

print(result.report)

In [ ]:
result.predictions.head()

## 13.2 Inspecting the agent execution log

Every agent logs its actions to `result.context.history`.

In [ ]:
for entry in result.context.history:
    print(f"[{entry['agent']:>12}]  {entry['message']}")

## 13.3 Using individual agents

You can use each agent independently for more control.

In [ ]:
from polars_ts.agents import CuratorAgent

curator = CuratorAgent()
report = curator.curate(df)
print(f"Observations:   {report.n_observations}")
print(f"Missing:        {report.n_missing}")
print(f"Outliers:       {report.n_outliers}")
print(f"Period:         {report.detected_period}")
print(f"Trend:          {report.has_trend}")
print(f"Stationary:     {report.is_stationary}")
print(f"Lookback rec.:  {report.recommended_lookback}")

In [ ]:
# Clean the data (impute + clip outliers)
cleaned = curator.curate_and_clean(df)
print(f"Nulls after cleaning: {cleaned['y'].null_count()}")
print(f"NaNs after cleaning:  {cleaned['y'].is_nan().sum()}")

### Lookback window trimming

The curator can detect regime changes and trim to the most recent stable regime.

In [ ]:
# Build a series with a regime change
rng2 = np.random.default_rng(99)
regime_vals = np.concatenate([10 + rng2.normal(0, 1, 90), 50 + rng2.normal(0, 5, 90)])
regime_dates = [datetime.date(2023, 1, 1) + datetime.timedelta(days=i) for i in range(180)]
regime_df = pl.DataFrame({"unique_id": ["regime"] * 180, "ds": regime_dates, "y": regime_vals.tolist()})

curator2 = CuratorAgent()
report2 = curator2.curate(regime_df)
print(f"Stationary: {report2.is_stationary}")
print(f"Recommended lookback: {report2.recommended_lookback}")

trimmed = curator2.trim_lookback(regime_df)
print(f"Original rows: {len(regime_df)}, Trimmed rows: {len(trimmed)}")

## 13.4 Planner → Forecaster with ensemble

When 3+ candidate models are selected, the planner enables ensemble mode.
The forecaster then combines predictions using inverse-MAE weights.

In [ ]:
from polars_ts.agents import ForecasterAgent, PlannerAgent

planner = PlannerAgent(horizon=14)
plan = planner.plan(cleaned, report)
print(f"Candidates: {plan.candidates}")
print(f"Ensemble:   {plan.ensemble}")
print(f"Rationale:  {plan.rationale}")

In [ ]:
forecaster = ForecasterAgent()
fc_result = forecaster.forecast(cleaned, plan)
print(f"Best model:       {fc_result.best_model}")
print(f"Model scores:     {fc_result.model_scores}")
if fc_result.ensemble_weights:
    print(f"Ensemble weights: {fc_result.ensemble_weights}")
fc_result.predictions.head()

## 13.5 Event context and automatic lookback trimming

Pass anticipated events and enable automatic lookback trimming via `TimeSeriesScientist`.

In [ ]:
events = [
    {"date": "2023-07-04", "description": "Independence Day — expected demand spike"},
    {"date": "2023-07-15", "description": "Summer sale begins"},
]

scientist2 = TimeSeriesScientist(
    horizon=14,
    events=events,
    trim_lookback=True,
)
result2 = scientist2.run(regime_df)

for entry in result2.context.history:
    print(f"[{entry['agent']:>12}]  {entry['message']}")

## 13.6 Custom LLM backend

Implement the `LLMBackend` protocol to plug in any provider (OpenAI, Anthropic, local models).
The agents will use `complete()` for enhanced diagnostics, rationale, and narrative reports.

In [ ]:
from polars_ts.agents import LLMBackend


class EchoBackend:
    """Example backend that echoes prompts (replace with real API calls)."""

    def complete(self, prompt: str) -> str:
        # In production: call openai.chat.completions.create(...) or anthropic.messages.create(...)
        return f"[LLM] Received {len(prompt)} chars"


# Verify it satisfies the protocol
assert isinstance(EchoBackend(), LLMBackend)

scientist3 = TimeSeriesScientist(horizon=7, backend=EchoBackend())
result3 = scientist3.run(df)
# The report now includes LLM-enhanced sections
print(result3.report[:500])

## 13.7 Multi-series forecasting

The framework handles multiple series automatically — diagnostics, planning, and
forecasting are applied per-group.

In [ ]:
# Create 3 series with different characteristics
n_multi = 100
multi_dates = [datetime.date(2023, 1, 1) + datetime.timedelta(days=i) for i in range(n_multi)]
rng3 = np.random.default_rng(7)

trending = np.linspace(5, 40, n_multi) + rng3.normal(0, 1, n_multi)
seasonal = 20 + 6 * np.sin(2 * np.pi * np.arange(n_multi) / 7) + rng3.normal(0, 1, n_multi)
flat = 15 + rng3.normal(0, 2, n_multi)

multi_df = pl.DataFrame({
    "unique_id": ["trending"] * n_multi + ["seasonal"] * n_multi + ["flat"] * n_multi,
    "ds": multi_dates * 3,
    "y": trending.tolist() + seasonal.tolist() + flat.tolist(),
})

scientist4 = TimeSeriesScientist(horizon=7)
result4 = scientist4.run(multi_df)
print(f"Total prediction rows: {len(result4.predictions)}")
result4.predictions.group_by("unique_id").agg(pl.col("y_hat").mean().alias("mean_forecast"))